# FX-AGENT — Milestone 3 (₹0 cloud installer)

**For Google Colab on Chromebook; no Linux, VS Code, paid AI APIs, or broker connection.**

This notebook installs **additive** M3 modules on a clean clone of `unknownopensource/FX-AGENT`, runs the unchanged M1/M2 tests plus M3 tests, and optionally publishes **only code** to a new GitHub review branch. It does **not** modify `main` or publish historical prices or secrets. Read `docs/MILESTONE_3.md` before using real data.

**Run cells top-to-bottom.** The first three are required for verification. Real-data import is optional and requires a legally obtained historical *bid/ask tick* CSV. The GitHub publishing cell requires `GITHUB_TOKEN` in Colab's **Secrets** panel with access to this repo (`Contents: Read and write`). Publishing always asks for a deliberate confirmation.

**Research-only:** FX-AGENT still has no live trading, broker API, realistic tick-level fills, or validated profits. Historical backtests are not investment advice.

## Cell 1 — Upload and verify the Milestone 3 ZIP

Download `FX_AGENT_Milestone3_Addon.zip` from ChatGPT, run this cell and select the ZIP **once**. It checks the expected file inventory and embedded SHA-256 hashes. This checks accidental corruption, not authenticity against a maliciously replaced ZIP.

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import zipfile, json, hashlib, uuid, stat

uploads = files.upload()
zip_candidates = [name for name in uploads if name.endswith('.zip')]
if len(zip_candidates) != 1:
    raise ValueError('Upload exactly one FX-AGENT Milestone 3 ZIP')
zip_name = zip_candidates[0]

with zipfile.ZipFile(zip_name) as z:
    actual = {item.filename for item in z.infolist() if not item.is_dir()}
    manifest = json.loads(z.read('MILESTONE3_MANIFEST.json'))
    expected = set(manifest['files']) | {'MILESTONE3_MANIFEST.json'}
    if actual != expected:
        raise ValueError(f'Unexpected ZIP inventory: missing={expected - actual}; extras={actual - expected}')
    if sum(item.file_size for item in z.infolist()) > 3_000_000:
        raise ValueError('Unexpectedly large ZIP')
    for item in z.infolist():
        path = PurePosixPath(item.filename)
        if path.is_absolute() or '..' in path.parts or '\\' in item.filename:
            raise ValueError('Unsafe path in ZIP')
        if (item.external_attr >> 16) & 0o170000 == stat.S_IFLNK:
            raise ValueError('Symlinks are forbidden')
    for name, digest in manifest['files'].items():
        if hashlib.sha256(z.read(name)).hexdigest() != digest:
            raise ValueError('File hash mismatch: ' + name)
    addon_root = Path('/content') / ('FX-AGENT-M3-ADDON-' + uuid.uuid4().hex[:8])
    addon_root.mkdir(parents=True, exist_ok=False)
    for item in z.infolist():
        target = (addon_root / item.filename).resolve()
        if not target.is_relative_to(addon_root.resolve()):
            raise ValueError('Unsafe ZIP extraction target')
    z.extractall(addon_root)
addon_files = tuple(sorted(manifest['files'])) + ('MILESTONE3_MANIFEST.json',)
print('PASS: Milestone 3 ZIP integrity checks')
print('New files:', len(addon_files))
print('Local ZIP directory:', addon_root)


## Cell 2 — Clone your public `main` and test the existing foundation

This is a **new directory** in Colab; it does not change your GitHub repository. Existing M1/M2 tests must pass before any M3 modules are installed.

In [ ]:
import subprocess, sys, hashlib
from pathlib import Path

REPO = 'unknownopensource/FX-AGENT'
repo_root = Path('/content') / ('FX-AGENT-M3-VERIFY-' + uuid.uuid4().hex[:8])
subprocess.run(['git','clone','--quiet','--depth','1','--branch','main',
                f'https://github.com/{REPO}.git',str(repo_root)],check=True)
base_sha = subprocess.check_output(['git','rev-parse','HEAD'], cwd=repo_root,text=True).strip()
required_old = ['fxagent/data.py','fxagent/paper.py','fxagent/timeframes.py',
                'tests/test_foundation.py','tests/test_milestone2.py']
if not all((repo_root / name).is_file() for name in required_old):
    raise RuntimeError('main is missing Milestone 1/2 files; do not continue')
tracked = subprocess.check_output(['git','ls-files','-z'],cwd=repo_root).split(b'\0')
tracked = [x.decode('utf-8') for x in tracked if x]
original_hashes = {p:hashlib.sha256((repo_root/p).read_bytes()).hexdigest() for p in tracked}
old_tests = subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],
                            cwd=repo_root, text=True, capture_output=True)
print(old_tests.stdout)
print(old_tests.stderr)
old_tests.check_returncode()
print('PASS: Milestones 1 and 2 baseline tests')
print('Original main commit:', base_sha)


## Cell 3 — Install additive modules, run all tests, and run the synthetic demo

Fails if an existing file has the same path but different content. Verifies **all originally tracked files retain identical SHA-256 hashes**. Demonstration uses synthetic data only; `data/` and `reports/` are not published.

In [ ]:
import shutil
for name in addon_files:
    source, target = addon_root/name, repo_root/name
    if target.exists():
        if target.read_bytes() != source.read_bytes():
            raise RuntimeError('Refusing to overwrite existing file: ' + name)
        continue
    target.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(source,target)

for filename, digest in original_hashes.items():
    if hashlib.sha256((repo_root/filename).read_bytes()).hexdigest() != digest:
        raise AssertionError('Milestone 1/2 tracked file changed: '+filename)

all_tests = subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],
                           cwd=repo_root,text=True,capture_output=True)
print(all_tests.stdout)
print(all_tests.stderr)
all_tests.check_returncode()
print('PASS: Full regression suite')

demo = subprocess.run([sys.executable,'-m','fxagent.demo_m3'],
                      cwd=repo_root,text=True,capture_output=True)
print(demo.stdout)
print(demo.stderr)
demo.check_returncode()

for filename, digest in original_hashes.items():
    if hashlib.sha256((repo_root/filename).read_bytes()).hexdigest() != digest:
        raise AssertionError('Milestone 1/2 file modified by demo: '+filename)

m3_verified = True
print('PASS: M3 synthetic demo and original source preservation')
print('Ready to open a GitHub review branch (optional publishing cell below).')


## Optional Cell 4 — Research on your own historical bid/ask tick CSV

Run **only if you have real tick prices** and have checked your source provider's terms. The CSV stays inside temporary Colab storage. The standardized format is `timestamp,bid,ask` (ISO UTC); `dukascopy-node` tick CSV `timestamp,askPrice,bidPrice,askVolume,bidVolume` (Unix milliseconds) is also supported. **M15 bid-only OHLC will be rejected**; there is no honest way to infer its historical spread. The data audit may refuse poor-quality input. Set your *final holdout start* before running; do not modify this cutoff after looking at results.

Example free historical export reference: https://www.dukascopy.com/swiss/english/marketwatch/historical/. This notebook does not download or redistribute provider data.

In [ ]:
# OPTIONAL: Execute only when you have a genuine historical bid/ask CSV.
from google.colab import files
import subprocess, sys

print('Upload one historical bid/ask tick CSV (not M15 bid-only OHLC).')
real_files = files.upload()
if len(real_files) != 1 or not next(iter(real_files)).lower().endswith('.csv'):
    raise ValueError('Upload exactly one CSV file')
filename = str(Path('/content')/next(iter(real_files)))
symbol = input('Forex pair EURUSD/GBPUSD/USDJPY/AUDUSD/USDCHF: ').strip().upper()
provider = input('Historical data provider name: ').strip()
source_url = input('Provider export/source URL: ').strip()
license_note = input('Your data-use / permission note: ').strip()
test_start = input('FINAL holdout start (UTC M15, e.g. 2024-05-01T00:00:00Z): ').strip()

run = subprocess.run([
    sys.executable,'-m','fxagent.run_m3',
    '--input',filename,'--symbol',symbol,'--provider',provider,
    '--source-url',source_url,'--permission-note',license_note,
    '--test-start',test_start,
    '--train-bars','960','--validation-bars','480','--step-bars','480',
],cwd=repo_root,text=True,capture_output=True)
print(run.stdout)
print(run.stderr)
run.check_returncode()
print('Reports are local Colab files only; do not publish them to your public repo.')


## Cell 5 — Publish **source files only** to a GitHub review branch

Run this after Cell 3 succeeded. Your `GITHUB_TOKEN` must be saved in Colab Secrets with **Contents: Read and write** for `FX-AGENT`. This cell refuses to write if `main` changed since the clone, and will never push to `main`. It publishes only the fixed M3 source/test/doc/notebook/manifest allowlist; no credentials or market-data files. It asks for `PUBLISH` before making any change.

In [ ]:
from google.colab import userdata
from urllib.parse import quote
import requests, base64, uuid

if globals().get('m3_verified') is not True:
    raise RuntimeError('Run successful tests and demo in Cell 3 before publishing')
key = userdata.get('GITHUB_TOKEN')
if not key:
    raise RuntimeError('Save GITHUB_TOKEN in Colab Secrets and enable notebook access')

api=f'https://api.github.com/repos/{REPO}'
session=requests.Session()
session.headers.update({'Authorization':f'Bearer {key}',
                        'Accept':'application/vnd.github+json',
                        'X-GitHub-Api-Version':'2022-11-28'})
info=session.get(api,timeout=30)
info.raise_for_status()
if info.json().get('full_name','').lower() != REPO.lower() or info.json().get('default_branch') != 'main':
    raise RuntimeError('Unexpected repository or default branch; no changes made')
head=session.get(f'{api}/git/ref/heads/main',timeout=30)
head.raise_for_status()
if head.json()['object']['sha'] != base_sha:
    raise RuntimeError('main changed after clone. Restart with a fresh clone to review differences.')

# Fail before creating a branch if any file path would overwrite existing content.
for name in addon_files:
    result=session.get(f'{api}/contents/{quote(name,safe="/")}',
                       params={'ref':'main'},timeout=30)
    if result.status_code == 200:
        raise RuntimeError(f'File already exists on main: {name}. Review the existing code manually.')
    if result.status_code != 404:
        result.raise_for_status()

print('Repo:',REPO)
print('Branch to be based on:',base_sha)
print('Additive files to publish (no datasets, reports, or secrets):')
for name in addon_files: print(' •',name)
if input('Type PUBLISH to create the review branch and upload source: ').strip() != 'PUBLISH':
    print('CANCELLED: no GitHub changes made')
else:
    review_branch='milestone-3-review-'+uuid.uuid4().hex[:8]
    branch_resp=session.post(f'{api}/git/refs',json={
        'ref':f'refs/heads/{review_branch}', 'sha':base_sha},timeout=30)
    branch_resp.raise_for_status()
    for name in addon_files:
        body=base64.b64encode((addon_root/name).read_bytes()).decode('ascii')
        response=session.put(f'{api}/contents/{quote(name,safe="/")}',json={
            'message':f'Add Milestone 3: {name}', 'content':body,
            'branch':review_branch},timeout=45)
        response.raise_for_status()
        print('Added:',name)
    compare=f'https://github.com/{REPO}/compare/main...{review_branch}?expand=1'
    print('SUCCESS: uploaded M3 to review branch only')
    print('OPEN AND REVIEW THIS URL:',compare)
    print('Create a pull request, review all files, then merge in GitHub if approved.')


## After the pull request is merged

Start a new Colab runtime and clone `main`; run both demonstrations and the full test suite:

```python
import subprocess, sys
folder='/content/FX-AGENT-M3-FINAL'
subprocess.run(['git','clone','--depth','1',
                'https://github.com/unknownopensource/FX-AGENT.git',folder],check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=folder,check=True)
subprocess.run([sys.executable,'-m','fxagent.demo_m3'],cwd=folder,check=True)
```

**Milestone 3 is not a self-running AI trader.** This verifies data intake and walk-forward research tooling only. The final holdout stays untouched, and no profitability has been established.